# MODIS + VIIRS harmonization — Steps 6 to 9 (DRAFT)

**Status: draft written before the real data arrived. It contains no results.**

It needs `monthly_counts.csv` from the Steps 2 to 5 notebook, made with `SAMPLE = False`.
Run one cell at a time. Read each "Check" note first.

The reasons behind each choice are in the project doc `harmonization-plan.md` (decisions D8 to D11).

**AI use:** the code in this notebook was drafted with Claude (Anthropic), an AI tool. Declare this on the project page.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Okabe-Ito colour-blind-safe palette. Lines also differ by style, not colour alone.
BLUE, ORANGE, VERMILLION, GREEN, GREY = "#0072B2", "#E69F00", "#D55E00", "#009E73", "#666666"

df = pd.read_csv("monthly_counts.csv")
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values("date").reset_index(drop=True)
print(len(df), "months,", df["date"].min().date(), "to", df["date"].max().date())
print(df.head().to_string())

## Split the overlap into training and test years

**What and why.** We learn ratios on Feb 2012 to Dec 2018. We test on 2019 to 2025. The test years are never used for learning. January 2012 is dropped because VIIRS began on the 20th.

**Check.** Training should be 83 months. Test should be 84 months. If not, some months are missing.

In [ ]:
OVERLAP_START = "2012-02-01"
TRAIN_END = "2018-12-31"
TEST_START, TEST_END = "2019-01-01", "2025-12-31"

both = df.dropna(subset=["modis_raw", "viirs"])
both = both[both["date"] >= OVERLAP_START]
train = both[both["date"] <= TRAIN_END]
test = both[(both["date"] >= TEST_START) & (both["date"] <= TEST_END)]
print("training months:", len(train), "(expect 83)")
print("test months:    ", len(test), "(expect 84)")
print("training totals: MODIS", int(train["modis_raw"].sum()), " VIIRS", int(train["viirs"].sum()))

## Step 6 — Compute the ratios, three ways

**What and why.** Three ways to turn a MODIS count into a VIIRS-equivalent count. We fit all three on the training years only.

1. **Overall ratio.** Total VIIRS divided by total MODIS. One number. Stable, ignores seasons.
2. **Ratio per calendar month.** Twelve numbers. Can follow the seasons, but each comes from only 7 months of data.
3. **Straight-line fit.** `viirs = a * modis + b`. The `b` allows VIIRS fire when MODIS sees none. Big haze months pull the line hard.

We use *ratio of sums*, not the average of monthly ratios. A month with 1 MODIS detection would otherwise swing the result.

**Check.**
- Is the overall ratio above 1? VIIRS should see more.
- Other teams report roughly 2 to 6 for other regions (unverified, not Sabah). A value like 0.5 or 50 means something is broken.
- Do the twelve monthly ratios jump around wildly? That suggests too little data per month.

**Common mistake.** Fitting on all years, then "testing" on the same years. That always looks good and proves nothing.

In [ ]:
MIN_MODIS_PER_MONTH = 20   # fewer MODIS detections than this in a calendar month -> use the overall ratio

def fit(data):
    model = {}
    model["overall"] = data["viirs"].sum() / data["modis_raw"].sum()
    sums = data.groupby("month")[["modis_raw", "viirs"]].sum()
    per_month = sums["viirs"] / sums["modis_raw"].where(sums["modis_raw"] >= MIN_MODIS_PER_MONTH)
    model["monthly"] = per_month.reindex(range(1, 13)).fillna(model["overall"])
    model["fallback_months"] = [m for m in range(1, 13) if not (sums["modis_raw"].get(m, 0) >= MIN_MODIS_PER_MONTH)]
    model["a"], model["b"] = np.polyfit(data["modis_raw"], data["viirs"], 1)
    return model

def predict(model, data, method):
    if method == "none":
        out = data["modis_raw"].astype(float)
    elif method == "overall":
        out = data["modis_raw"] * model["overall"]
    elif method == "monthly":
        out = data["modis_raw"] * data["month"].map(model["monthly"])
    elif method == "line":
        out = (model["a"] * data["modis_raw"] + model["b"]).clip(lower=0)
    else:
        raise ValueError(method)
    return out

m_train = fit(train)
print("overall ratio:", round(m_train["overall"], 2))
print("line fit: viirs =", round(m_train["a"], 2), "* modis +", round(m_train["b"], 1))
print("ratio per calendar month:")
print(m_train["monthly"].round(2).to_string())
if m_train["fallback_months"]:
    print("months that fell back to the overall ratio (too few MODIS detections):", m_train["fallback_months"])
if not (1 < m_train["overall"] < 15):
    print("ODD: overall ratio is outside 1 to 15. Check the cleaning steps before trusting anything below.")

### Is the ratio steady from year to year?

**What and why.** A ratio only works if the two sensors keep the same relationship. MODIS orbits began drifting around 2020 to 2022, which could change it. This table is ratio of sums per year, across all overlap years.

**Check.** A slow climb or drop after 2021 is worth reporting to judges. One odd year is probably weather. A trend is the sensor.

In [ ]:
yearly = both.groupby("year")[["modis_raw", "viirs"]].sum()
yearly["ratio"] = (yearly["viirs"] / yearly["modis_raw"]).round(2)
print(yearly.to_string())

## Step 8 — Hold-out test

*(Step 8 comes before Step 7 here. We test first, then build the final series with the winner.)*

**What and why.** In the test years we pretend VIIRS does not exist. We predict it from MODIS, then compare with the real VIIRS. "none" is the raw join with no fix.

- **MAE**: average size of the miss, in hotspots per month. Lower is better.
- **bias**: average of prediction minus actual. Positive means we over-predict.
- **total_error_pct**: all misses added up, as a percent of all real VIIRS detections.

**Check.**
- "none" should be far worse than the rest. If not, the sensors barely differ here, which would be odd.
- Compare the three fixes with each other. That is the real contest.
- If "monthly" is worse than "overall", the monthly ratios are too noisy. Use overall and say why.

**Common mistake.** Picking the method by looking at the test chart, then calling the test "independent". Pick by the table, once.

In [ ]:
rows = []
for method in ["none", "overall", "monthly", "line"]:
    pred = predict(m_train, test, method)
    err = pred - test["viirs"]
    rows.append({
        "method": method,
        "MAE": round(err.abs().mean(), 1),
        "bias": round(err.mean(), 1),
        "total_error_pct": round(100 * err.abs().sum() / test["viirs"].sum(), 1),
    })
results = pd.DataFrame(rows).set_index("method")
print(results.to_string())

err_by_year = pd.DataFrame({
    m: (predict(m_train, test, m) - test["viirs"]).abs().groupby(test["year"]).mean().round(1)
    for m in ["overall", "monthly", "line"]
})
print()
print("MAE by test year (does the error grow in later years?):")
print(err_by_year.to_string())

### Choose the method

Set `METHOD` to the best one from the table above: `"overall"`, `"monthly"` or `"line"`.
Write one sentence in your notes saying why. Judges will ask.

In [ ]:
METHOD = "monthly"   # change this after reading the results table

### Chart 2 — test years: real VIIRS vs MODIS x ratio

**Check.** The two lines should rise and fall together. Look at the biggest peak. Does the prediction reach it, overshoot it, or fall short?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(test["date"], test["viirs"], color=BLUE, lw=2, label="VIIRS actual")
ax.plot(test["date"], predict(m_train, test, METHOD), color=ORANGE, lw=2, ls="--", label=f"MODIS adjusted ({METHOD})")
ax.plot(test["date"], test["modis_raw"], color=GREY, lw=1.2, ls=":", label="MODIS raw (no fix)")
ax.set_title("Hold-out test, Sabah: ratios learned on 2012-2018, tested on 2019-2025")
ax.set_ylabel("Hotspots per month")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("chart_holdout_test.png", dpi=150)
plt.show()

## Step 7 — Build the harmonized series

**What and why.** Now that the method is tested, we refit it on *all* overlap years. More data gives a steadier ratio. The hold-out numbers above stay as our honest error estimate.

- `modis_adjusted` = MODIS scaled into VIIRS-equivalent counts, for every month.
- `harmonized` = real VIIRS from Feb 2012 onward, `modis_adjusted` before that. Real measurements beat estimates where they exist.

**Check.**
- `harmonized` has no empty values.
- Before 2012, `harmonized` equals `modis_adjusted`. From Feb 2012, it equals `viirs`.
- Months before July 2002 are flagged `terra_only`. MODIS had one satellite then, so those values are probably too low.

**Common mistake.** Averaging VIIRS and adjusted MODIS in the overlap. That blends a measurement with an estimate for no gain.

In [ ]:
m_final = fit(both)
print("final overall ratio (all overlap years):", round(m_final["overall"], 2))

out = df.copy()
out["modis_adjusted"] = predict(m_final, out, METHOD).round(1)
use_viirs = (out["date"] >= OVERLAP_START) & out["viirs"].notna()
out["harmonized"] = np.where(use_viirs, out["viirs"], out["modis_adjusted"])
out["terra_only"] = out["date"] < "2002-07-01"

print("empty harmonized values:", int(out["harmonized"].isna().sum()), "(expect 0)")
print(out[["year", "month", "modis_raw", "viirs", "modis_adjusted", "harmonized"]].iloc[132:140].to_string())

### Chart 1 — raw join vs harmonized

**What it shows.** The raw join is MODIS until Jan 2012, then VIIRS. It should show a step up in 2012. The harmonized line should not.

**Check.** Look at the years right around the dashed line. Is the step gone? If the harmonized line still jumps, the ratio is too small. If it drops, too big.

In [ ]:
raw_join = np.where(use_viirs, out["viirs"], out["modis_raw"])
annual = pd.DataFrame({"year": out["year"], "raw_join": raw_join, "harmonized": out["harmonized"]}).groupby("year").sum()

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
axes[0].plot(out["date"], raw_join, color=VERMILLION, lw=1.4, ls="--", label="Raw join (no fix)")
axes[0].plot(out["date"], out["harmonized"], color=BLUE, lw=1.6, label="Harmonized")
axes[0].axvline(pd.Timestamp(OVERLAP_START), color=GREY, ls=":", lw=1)
axes[0].set_title("Sabah monthly hotspots: raw join vs harmonized (dotted line = VIIRS starts)")
axes[0].set_ylabel("Hotspots per month")
axes[0].legend(frameon=False)

w = 0.4
axes[1].bar(annual.index - w / 2, annual["raw_join"], width=w, color=VERMILLION, hatch="//", edgecolor="white", label="Raw join (no fix)")
axes[1].bar(annual.index + w / 2, annual["harmonized"], width=w, color=BLUE, label="Harmonized")
axes[1].axvline(2011.5, color=GREY, ls=":", lw=1)
axes[1].set_title("Same data, yearly totals (first and last years may be partial)")
axes[1].set_ylabel("Hotspots per year")
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("chart_raw_vs_harmonized.png", dpi=150)
plt.show()

## Step 9 — "Now vs normal"

**What and why.** For each month: its harmonized count divided by the median of the same calendar month in *earlier* years. 1.0 means normal. 2.0 means twice the usual. We use the median because one haze year would drag a mean upward.

**Rules.** Only past years count. At least 5 past years are needed, else the value is empty. If normal is 0, the value is empty (no dividing by zero).

**Check.**
- The first 5 years are empty. That is correct.
- Look at the latest month. Does the answer match what you know about this year's fire season?
- Your team's haze years (2015, 2019) should show high values in the burning months. If they do not, say so. Do not force it.

**Common mistake.** Including the current year in its own "normal". Then every extreme month looks less extreme.

In [ ]:
MIN_PAST_YEARS = 5

out = out.sort_values("date").reset_index(drop=True)
out["normal_median"] = out.groupby("month")["harmonized"].transform(
    lambda s: s.expanding(min_periods=MIN_PAST_YEARS).median().shift(1)
)
out["now_vs_normal"] = (out["harmonized"] / out["normal_median"].where(out["normal_median"] > 0)).round(2)

latest = out.iloc[-1]
print("latest month:", int(latest["year"]), int(latest["month"]))
print("harmonized:", latest["harmonized"], " normal:", latest["normal_median"], " now vs normal:", latest["now_vs_normal"])
print()
print("ten most unusual months on record:")
print(out.nlargest(10, "now_vs_normal")[["year", "month", "harmonized", "normal_median", "now_vs_normal"]].to_string())

## Save the final CSV

`terra_only` is an extra flag column. Keep it for the limits section, or drop it for the website.

In [ ]:
final = out[["year", "month", "modis_raw", "viirs", "modis_adjusted", "harmonized", "normal_median", "now_vs_normal", "terra_only"]]
final.to_csv("sabah_harmonized_monthly.csv", index=False)
print("saved sabah_harmonized_monthly.csv -", len(final), "rows")
print(final.tail(6).to_string())